In [ ]:
import multiprocessing as mp
import pickle

import numpy as np
import torch
from matplotlib import pyplot as plt

from src.models import BinaryClassifier
from src.utils import get_filepath
from src.priors import build_priors, integrate_nd_vegas
from src.data import draw_data, get_data
from src.training import train, test_model
from src.plotting import (
    _enforce_markers,
    plot_all_rocs,
    plot_error_and_hpd_width,
    plot_errorbars,
    plot_prior_contours,
    plot_prior_sample_histograms,
    plot_ratio_violins,
    plot_reweighted_distributions,
    plot_training_tests,
)
from src.posterior import (
    create_inference_data,
    get_first_column_scatter_data,
    get_posteriors_and_errors,
)
from src.verification import calculate_all_ratios, reweight_distributions

plt.rc("axes", prop_cycle=plt.cycler(color=plt.get_cmap("Set1").colors))

In [ ]:
config = {
    "data": {
        "prior_args": {
            "uniform": None,
            "normal": (5, 4),
            "exponential": (0.1,),
            "grid": (0.2,),
        },
        "parameter_range": (0, 10),
        "std_dev": 2,
        "n_parameters": 4,
        "n_train": 70_000,
        "n_test": 15_000,
        "n_validation": 15_000,
    },
    "classifier": {
        "n_inputs": 4,
        "n_hidden_layers": 3,
        "n_units": 64,
    },
    "training": {
        "learning_rate": 0.001,
        "batch_size": 128,
        "n_epochs": 5,
        "model_iterations": 1,
    },
}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generator = np.random.default_rng(0)

In [ ]:
priors, prior_samplers, param_min, param_max = build_priors(config, generator)

#### Prior Distirbutions:

In [ ]:
plot_prior_contours(
    priors,
    parameter_range=config["data"]["parameter_range"],
    n_parameters=config["data"]["n_parameters"],
)

In [ ]:
plot_prior_sample_histograms(prior_samplers, config)

In [ ]:
for prior in priors:
    integral = integrate_nd_vegas(
        prior,
        [(param_min, param_max)] * config["data"]["n_parameters"],
        nitn=20,
        neval=10_000,
    )[0]
    print(f"Integral of {prior.__name__}: {integral:.5f}")

#### Data:

In [ ]:
example_train_set, example_validation_set, example_test_set = get_data(
    prior_samplers[0],
    config,
    generator,
    show_output=True,
)

#### Training:

#### Testing:

#### Plots:

#### Run:

In [ ]:
models = [{} for _ in range(config["training"]["model_iterations"])]
test_results = [{} for _ in range(config["training"]["model_iterations"])]

for prior_sampler in prior_samplers:
    prior_name = prior_sampler.__name__
    for i in range(config["training"]["model_iterations"]):
        train_set, validation_set, test_set = get_data(
            prior_sampler,
            config,
            generator,
            show_output=False,
        )
        model = BinaryClassifier(config)
        model, training_results = train(
            model,
            *train_set,
            *validation_set,
            config=config,
            device=device,
            show_output=False,
        )
        filename_plots = get_filepath(f"training/training_{i}/training_{prior_name}_prior.svg", config)
        filename_model = get_filepath(f"models/models_{i}/model_{prior_name}_prior.pth", config)
        torch.save(model.state_dict(), filename_model)
        test_result = test_model(model, *test_set, device=device, show_output=False)
        plot_training_tests(training_results, test_result, f"{i}. {prior_name}", filename_plots)
        models[i][prior_name] = model
        test_results[i][prior_name] = test_result

plot_all_rocs(test_results, filename=get_filepath("training/all_roc_curves.svg", config))

#### Posterior:

In [ ]:
save = True
n_repititions_per_parameter = 7
n_parameters = int(2e2)

posterior_qmc_samples = 2**16
posterior_eval_batch_size = 2**15
posterior_max_model_evals_per_prior = int(2e15)
posterior_qmc_seed = 2026

parameters_min_max = config["data"]["parameter_range"]
all_parameters_in_range = [
    np.linspace(parameters_min_max[0], parameters_min_max[1], n_parameters)
    for _ in range(config["data"]["n_parameters"])
]
print(f"GPU: {device.type}; CPUs: {mp.cpu_count()}")

filename_errorbars = get_filepath("posterior_errors/errorbars.svg", config) if save else None
filename_errorbars_ratios = get_filepath("posterior_errors/errorbars_ratios.svg", config) if save else None
filename_error_and_hpd_width = get_filepath("posterior_errors/error_and_hpd_width.svg", config) if save else None
filename_error_and_hpd_width_ratios = get_filepath("posterior_errors/error_and_hpd_width_ratios.svg", config) if save else None
filename_posteriors_data = get_filepath("posterior_errors/results_data.pkl", config) if save else None

In [ ]:
inference_data = create_inference_data(
    parameters_min_max,
    config,
    generator,
    n_parameters_to_infer_per_dim=20,
    margin=0,
    n_repititions_per_parameter=n_repititions_per_parameter,
)
all_posteriors, all_ratios, all_HPDs_posterior, all_HPDs_ratio = get_posteriors_and_errors(
    inference_data,
    all_parameters_in_range,
    models=models,
    priors=priors,
    device=device,
    n_qmc_samples=posterior_qmc_samples,
    eval_batch_size=posterior_eval_batch_size,
    max_model_evals_per_prior=posterior_max_model_evals_per_prior,
    qmc_seed=posterior_qmc_seed,
)

if save:
    with open(filename_posteriors_data, "wb") as handle:
        pickle.dump(
            {
                "all_HPDs_posterior": all_HPDs_posterior,
                "all_HPDs_ratio": all_HPDs_ratio,
                "inference_data": inference_data,
                "all_posteriors": all_posteriors,
                "all_ratios": all_ratios,
                "all_parameters_in_range": all_parameters_in_range,
                "n_repititions_per_parameter": n_repititions_per_parameter,
                "n_parameters": n_parameters,
            },
            handle,
        )

In [ ]:
plot_errorbars(
    all_HPDs_posterior,
    inference_data,
    all_posteriors,
    all_parameters_in_range,
    show_annotations=False,
    filename=filename_errorbars,
)
plot_errorbars(
    all_HPDs_ratio,
    inference_data,
    all_ratios,
    all_parameters_in_range,
    show_annotations=False,
    filename=filename_errorbars_ratios,
)

fig, _ = plot_error_and_hpd_width(
    all_HPDs_posterior,
    inference_data,
    all_posteriors,
    all_parameters_in_range,
    filename=filename_error_and_hpd_width,
)
_enforce_markers(fig, filename=filename_error_and_hpd_width)

fig, _ = plot_error_and_hpd_width(
    all_HPDs_ratio,
    inference_data,
    all_ratios,
    all_parameters_in_range,
    filename=filename_error_and_hpd_width_ratios,
)
_enforce_markers(fig, filename=filename_error_and_hpd_width_ratios)

In [ ]:
averaged_bias_error = get_first_column_scatter_data(
    inference_data,
    all_ratios,
    all_parameters_in_range,
    all_HPDs_ratio,
)
averaged_bias_error

#### Posterior Checks:

Check I: ratio expectation value

$1=\int p(x|\theta_0) dx = \int r(x|\theta_0) p(x) dx = \mathbb{E} [r(x|\theta_0)]$

i.e. if $\mathbb{E} [r(x|\theta_0)]=1$, then $r(x|\theta_0) = p(x|\theta_0) / p(x) = p(\theta_0 | x) / p(\theta_0)$ is plausible

In [ ]:
n_ratio_samples = 1_000
verification_parameters = generator.uniform(
    low=param_min,
    high=param_max,
    size=(n_ratio_samples, config["data"]["n_parameters"]),
)
verification_data = draw_data(verification_parameters, config, generator)
test_parameters = np.linspace(param_min, param_max, 5)
all_ratio_checks = calculate_all_ratios(
    verification_data,
    test_parameters,
    models=models,
    priors=priors,
    device=device,
)

filename_ratio_violins = get_filepath("verifications/ratio_violins.svg", config) if save else None
plot_ratio_violins(all_ratio_checks, test_parameters, filename=filename_ratio_violins)

#### Check II: reweighting

$R_{10}(x) = \frac{r(x|\theta_1)}{r(x|\theta_0)} = \frac{p(x|\theta_1)}{p(x|\theta_0)}$

if $p(x|\theta_1) = R_{10}(x)p(x|\theta_0)$: likelihood ratio trick holds

For the nD case below, the ratios use the full data vector while the plot visualizes the first observed component.

In [ ]:
n_test_data = int(1e6)
n_bins = int(1e3)
test_parameter_sets = [(3, 7)]

for i, model_arr in enumerate(models):
    for test_parameter_set in test_parameter_sets:
        data_x, reweighted_distributions, reweighting_distributions, all_outs = reweight_distributions(
            test_parameter_set,
            model_arr,
            priors=priors,
            config=config,
            generator=generator,
            device=device,
            n_test_data=n_test_data,
            n_bins=n_bins,
            projection_dim=0,
        )
        filename_reweighted_distributions = get_filepath(
            f"verifications/reweighting_{i}/reweighted_distributions_{test_parameter_set[0]}_{test_parameter_set[1]}.svg",
            config,
        ) if save else None
        plot_reweighted_distributions(
            priors,
            data_x,
            reweighted_distributions,
            test_parameter_set,
            n_bins,
            filename=filename_reweighted_distributions,
        )